## Threshold 4Å Re-baseline

Re-run feature extraction and correlation analysis using RMSD threshold = 4.0Å (previously 2.0Å).

**Top-feature selection:** For each of the 120 SAE features, we compute the Spearman
correlation between that feature's activation values and RMSD across all test-set poses.
The top 20 are the features with the **highest positive Spearman correlation with RMSD** —
i.e. features that tend to fire more as pose quality worsens (continuous relationship).
This is a different lens than the logistic regression below, which optimises for binary classification.

**Output files:**
- `processed_data/metadata_rmsd4.csv`
- `processed_data/features_for_paper.npz` — test-set activations for k=3, 8, 15
- `processed_data/top_features_k3/8/15.csv`
- `processed_data/roc_auc_comparison.png`

In [19]:
import warnings
import torch
import numpy as np
import pandas as pd
import sys
import os
from scipy.stats import spearmanr, ConstantInputWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=ConstantInputWarning)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_processor import load_processed_data
from src.model import TopKSAE

OUTPUT_DIR     = '/Users/bridget/Desktop/projects/laloo-sae/processed_data'
MODEL_DIR      = os.path.expanduser('~/Desktop/projects/laloo-sae/models/12_21_25')
RMSD_THRESHOLD = 4.0
K_VALUES       = [3, 8, 15]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Model dir:  {MODEL_DIR}')

Device: cpu
Output dir: /Users/bridget/Desktop/projects/laloo-sae/processed_data
Model dir:  /Users/bridget/Desktop/projects/laloo-sae/models/12_21_25


### 1. Load data and update quality labels

In [20]:
latents_normalized, metadata, stats = load_processed_data(OUTPUT_DIR)
print(f'Latents shape: {latents_normalized.shape}')
print(f'Metadata shape: {metadata.shape}')
metadata.head()

Loaded 435,069 samples from /Users/bridget/Desktop/projects/laloo-sae/processed_data
Latents shape: (435069, 30)
Metadata shape: (435069, 6)


,case_id,sample_idx,rmsd,energy,generation,global_idx
0,afab_2nnq_afab_3fr4,0,2.385954,-5425.862544,15,0
1,afab_2nnq_afab_3fr4,1,3.430689,10000.000000,15,1
2,afab_2nnq_afab_3fr4,2,3.147245,-5404.352427,8,2
3,afab_2nnq_afab_3fr4,3,2.383592,-5421.452032,14,3
4,afab_2nnq_afab_3fr4,4,3.503210,-5409.776425,8,4


In [21]:
metadata_4 = metadata.copy()
metadata_4['good_pose'] = metadata_4['rmsd'] < RMSD_THRESHOLD

n_good = metadata_4['good_pose'].sum()
n_bad  = (~metadata_4['good_pose']).sum()
print(f'RMSD threshold: {RMSD_THRESHOLD}Å')
print(f'  Good poses (RMSD < {RMSD_THRESHOLD}Å): {n_good:,} ({n_good/len(metadata_4)*100:.1f}%)')
print(f'  Bad  poses (RMSD >= {RMSD_THRESHOLD}Å): {n_bad:,} ({n_bad/len(metadata_4)*100:.1f}%)')

out_path = os.path.join(OUTPUT_DIR, 'metadata_rmsd4.csv')
metadata_4.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')

RMSD threshold: 4.0Å
  Good poses (RMSD < 4.0Å): 156,243 (35.9%)
  Bad  poses (RMSD >= 4.0Å): 278,826 (64.1%)

Saved: /Users/bridget/Desktop/projects/laloo-sae/processed_data/metadata_rmsd4.csv


### 2. Load train / test splits

In [22]:
splits    = np.load(os.path.join(OUTPUT_DIR, 'splits.npz'), allow_pickle=True)
train_idx = splits['train_idx']
test_idx  = splits['test_idx']
print(f'Train: {len(train_idx):,}  |  Test: {len(test_idx):,}')

# Binary labels: 1 = good pose, 0 = bad pose
y_train = metadata_4['good_pose'].values[train_idx].astype(int)
y_test  = metadata_4['good_pose'].values[test_idx].astype(int)

# Raw latents (30-dim) -- baseline for logistic regression
X_raw_train = latents_normalized[train_idx]
X_raw_test  = latents_normalized[test_idx]

# Metadata for test-set correlation analysis
metadata_test = metadata_4.iloc[test_idx].reset_index(drop=True)
rmsd_test     = metadata_test['rmsd'].values
energy_test   = metadata_test['energy'].values
good_mask     = metadata_test['good_pose'].values

print(f'Test good: {good_mask.sum():,} ({good_mask.mean()*100:.1f}%)')
print(f'Test bad:  {(~good_mask).sum():,} ({(~good_mask).mean()*100:.1f}%)')

Train: 294,851  |  Test: 68,614
Test good: 25,127 (36.6%)
Test bad:  43,487 (63.4%)


### 3. Load SAE models (k = 3, 8, 15)

In [23]:
summary = torch.load(
    os.path.join(MODEL_DIR, 'training_summary.pkl'),
    map_location='cpu', weights_only=False
)

trained_models = {}
for k in K_VALUES:
    run_stats    = summary[k]
    best_run_idx = int(np.argmin([r['best_val_loss'] for r in run_stats]))
    filepath     = os.path.join(MODEL_DIR, f'topksae_k{k}_run{best_run_idx}.pt')
    model = TopKSAE(input_dim=30, hidden_dim=120, k=k, auxk=12,
                    batch_size=256, dead_steps_threshold=2000).to(device)
    model.load_state_dict(
        torch.load(filepath, map_location=device, weights_only=False)
    )
    model.eval()
    trained_models[k] = model
    print(f'  [+] k={k}: run {best_run_idx}  '
          f'(val_loss={run_stats[best_run_idx]["best_val_loss"]:.4f})')

print(f'\nLoaded {len(trained_models)} models.')

  [+] k=3: run 4  (val_loss=0.3879)
  [+] k=8: run 4  (val_loss=0.2220)
  [+] k=15: run 2  (val_loss=0.0512)

Loaded 3 models.


### 4. Extract features (test + train) and save

In [24]:
BATCH_SIZE = 2048

def extract_activations(model, latents_np, batch_size=BATCH_SIZE):
    """Run model.get_acts() in batches. Input: numpy [N, 30]. Returns numpy [N, 120]."""
    all_acts = []
    for start in range(0, latents_np.shape[0], batch_size):
        batch = torch.tensor(latents_np[start:start + batch_size],
                             dtype=torch.float32, device=device)
        all_acts.append(model.get_acts(batch).cpu().numpy())
    return np.vstack(all_acts)


features_test  = {}   # correlation analysis + logistic regression eval
features_train = {}   # logistic regression fitting

for k in K_VALUES:
    print(f'k={k}:', end=' ', flush=True)
    features_test[k]  = extract_activations(trained_models[k], X_raw_test)
    print(f'test={features_test[k].shape}', end='  ', flush=True)
    features_train[k] = extract_activations(trained_models[k], X_raw_train)
    print(f'train={features_train[k].shape}  '
          f'zeros={(features_test[k]==0).mean()*100:.1f}%')

feat_path = os.path.join(OUTPUT_DIR, 'features_for_paper.npz')
np.savez_compressed(
    feat_path,
    k3=features_test[3],
    k8=features_test[8],
    k15=features_test[15],
    test_idx=test_idx,
)
print(f'\nSaved: {feat_path}')

k=3: test=(68614, 120)  train=(294851, 120)  zeros=97.5%
k=8: test=(68614, 120)  train=(294851, 120)  zeros=93.3%
k=15: test=(68614, 120)  train=(294851, 120)  zeros=87.5%

Saved: /Users/bridget/Desktop/projects/laloo-sae/processed_data/features_for_paper.npz


### 5. Compute correlations and activation rates → top-feature CSVs

In [25]:
def compute_feature_stats(acts_test, acts_train, y_train, rmsd, energy, good_mask, top_n=20):
    """
    For each feature column in acts [N, 120]:
      corr_rmsd              Spearman correlation with continuous RMSD
      corr_energy            Spearman correlation with energy
      activation_rate        fraction of poses where feature > 0
      good/bad_activation_rate  activation rate split by 4Å label
      probe_weight           coefficient from L1 logistic regression (bad-pose probe)
      probe_rank             rank by probe_weight (1 = most discriminative)

    Top features = 20 highest positive corr_rmsd (Spearman ranking).
    Probe ranking is included as a second, multivariate lens:
      - Spearman: univariate, continuous RMSD relationship per feature
      - L1 probe: multivariate, jointly discriminative after penalising redundancy
    """
    # Fit L1 probe on train activations
    probe = LogisticRegression(
        penalty='l1', solver='saga', max_iter=2000,
        class_weight='balanced', random_state=42
    )
    # y_train here is 0/1 for good/bad pose — probe predicts bad (class 1)
    y_bad_train = 1 - y_train   # flip so bad = 1, matching sae_all convention
    probe.fit(acts_train, y_bad_train)
    weights = probe.coef_[0]                         # shape [120]
    # rank: feature with highest weight = rank 1
    probe_ranks = len(weights) - np.argsort(np.argsort(weights))

    records = []
    for feat_idx in range(acts_test.shape[1]):
        v              = acts_test[:, feat_idx]
        corr_r,   _    = spearmanr(v, rmsd)
        corr_e,   _    = spearmanr(v, energy)
        active         = v > 0
        records.append({
            'feature':              feat_idx,
            'corr_rmsd':            corr_r,
            'corr_energy':          corr_e,
            'activation_rate':      active.mean(),
            'good_activation_rate': active[good_mask].mean()  if good_mask.sum()  > 0 else float('nan'),
            'bad_activation_rate':  active[~good_mask].mean() if (~good_mask).sum() > 0 else float('nan'),
            'probe_weight':         weights[feat_idx],
            'probe_rank':           int(probe_ranks[feat_idx]),
        })
    df  = pd.DataFrame(records)
    top = df.nlargest(top_n, 'corr_rmsd').reset_index(drop=True)
    return df, top, probe


all_feature_dfs = {}
probes = {}
for k in K_VALUES:
    print(f'\n--- k={k} ---')
    all_df, top_df, probe = compute_feature_stats(
        features_test[k], features_train[k], y_train,
        rmsd_test, energy_test, good_mask, top_n=20
    )
    all_feature_dfs[k] = all_df
    probes[k] = probe

    csv_path = os.path.join(OUTPUT_DIR, f'top_features_k{k}.csv')
    top_df.to_csv(csv_path, index=False)
    print(f'Saved: {csv_path}')
    print(top_df[['feature', 'corr_rmsd', 'probe_weight', 'probe_rank',
                  'activation_rate', 'good_activation_rate', 'bad_activation_rate']]
          .to_string(index=False, float_format='{:.4f}'.format))

    # Show top 5 by probe weight for comparison
    top_probe = all_df.nlargest(5, 'probe_weight')[['feature', 'probe_weight', 'probe_rank', 'corr_rmsd']]
    print(f'\n  Top 5 by probe weight (k={k}):')
    print(top_probe.to_string(index=False, float_format='{:.4f}'.format))


--- k=3 ---
Saved: /Users/bridget/Desktop/projects/laloo-sae/processed_data/top_features_k3.csv
 feature  corr_rmsd  probe_weight  probe_rank  activation_rate  good_activation_rate  bad_activation_rate
     107     0.3598        0.1123          41           0.0896                0.0098               0.1358
      70     0.2117       -0.6097         118           0.0234                0.0000               0.0369
      80     0.1992        0.0278          58           0.1968                0.1589               0.2188
      88     0.1625        0.0312          57           0.0730                0.0543               0.0838
      92     0.1536       -0.3053         106           0.0628                0.0375               0.0775
      38     0.1417        0.0958          44           0.0492                0.0203               0.0658
      32     0.1412       -0.2220          97           0.0557                0.0203               0.0761
      87     0.1339        0.3192          16          

### 6. Logistic regression — auPR & ROC-AUC: SAE features vs raw latents

Fit on train split, evaluate on test split.

- **Baseline & SAE probes** both use `class_weight='balanced'`
- **SAE probes** use `penalty='l1'` (consistent with sae_all.ipynb; sparsity in feature space)
- **Baseline** uses default L2 (30-dim input, L1 less meaningful at this dimensionality)
- **auPR** is the primary metric here (better than AUC for the imbalanced 64/36 bad/good split)
- ROC-AUC included for comparison with prior results

In [26]:
# Raw latent baseline: L2, balanced
clf_raw = LogisticRegression(max_iter=1000, solver='lbfgs', C=1.0,
                              class_weight='balanced', random_state=42)
clf_raw.fit(X_raw_train, y_train)
prob_raw = clf_raw.predict_proba(X_raw_test)[:, 1]

# SAE probes: L1, balanced (consistent with sae_all.ipynb)
clf_sae = {}
for k in K_VALUES:
    clf = LogisticRegression(penalty='l1', solver='saga', max_iter=2000, C=1.0,
                              class_weight='balanced', random_state=42)
    clf.fit(features_train[k], y_train)
    clf_sae[k] = clf

representations = {
    'Raw latents (30-dim)': (prob_raw,                                    X_raw_train.shape[1]),
    'SAE k=3  (120-dim)':   (clf_sae[3].predict_proba(features_test[3])[:, 1],  120),
    'SAE k=8  (120-dim)':   (clf_sae[8].predict_proba(features_test[8])[:, 1],  120),
    'SAE k=15 (120-dim)':   (clf_sae[15].predict_proba(features_test[15])[:, 1], 120),
}

# y_test = 1 good, 0 bad. auPR and AUC score the good-pose class.
# auPR for good pose = precision-recall for recovering good poses — directly relevant.
roc_results  = {}
aupr_results = {}
print(f'{"Representation":<30}  {"auPR":>8}  {"ROC-AUC":>8}  {"n_feat":>6}')
print('-' * 60)
for name, (probs, n_feat) in representations.items():
    aupr = average_precision_score(y_test, probs)
    auc  = roc_auc_score(y_test, probs)
    aupr_results[name] = aupr
    roc_results[name]  = auc
    print(f'{name:<30}  {aupr:>8.4f}  {auc:>8.4f}  {n_feat:>6}')

Representation                      auPR   ROC-AUC  n_feat
------------------------------------------------------------
Raw latents (30-dim)              0.6125    0.7146      30
SAE k=3  (120-dim)                0.3825    0.5529     120
SAE k=8  (120-dim)                0.3646    0.5013     120
SAE k=15 (120-dim)                0.5021    0.6210     120


In [27]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#888888', '#2196F3', '#4CAF50', '#FF5722']
names  = list(roc_results.keys())

# Left: auPR bar chart
ax = axes[0]
aupr_vals = [aupr_results[n] for n in names]
bars = ax.barh(names, aupr_vals, color=colors, edgecolor='white')
ax.set_xlim(0, 1.0)
ax.set_xlabel('auPR (average precision, good-pose class)')
ax.set_title('auPR by representation')
for bar, v in zip(bars, aupr_vals):
    ax.text(v + 0.01, bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=9)

# Middle: ROC-AUC bar chart
ax = axes[1]
auc_vals = [roc_results[n] for n in names]
bars = ax.barh(names, auc_vals, color=colors, edgecolor='white')
ax.set_xlim(0.5, 1.0)
ax.set_xlabel('ROC-AUC')
ax.set_title('ROC-AUC by representation')
for bar, v in zip(bars, auc_vals):
    ax.text(v + 0.003, bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=9)
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5)

# Right: ROC curves
ax = axes[2]
all_probs = [
    prob_raw,
    clf_sae[3].predict_proba(features_test[3])[:, 1],
    clf_sae[8].predict_proba(features_test[8])[:, 1],
    clf_sae[15].predict_proba(features_test[15])[:, 1],
]
for name, probs, color in zip(names, all_probs, colors):
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax.plot(fpr, tpr, label=f'{name} ({roc_results[name]:.3f})', color=color)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('ROC curves')
ax.legend(fontsize=7)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, 'roc_auc_comparison.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

Saved: /Users/bridget/Desktop/projects/laloo-sae/processed_data/roc_auc_comparison.png


### 7. Filter threshold analysis

Top 5 bad-pose features per k by Spearman correlation with RMSD (from section 5).
For each feature: activation distribution (good vs bad, log scale) + precision/recall/F1 threshold sweep.
Best-F1 threshold used for the ablation.


In [28]:
top_spearman = {k: all_feature_dfs[k].nlargest(5, "corr_rmsd")["feature"].astype(int).tolist()
                for k in K_VALUES}
print("Top 5 by Spearman:", top_spearman)

y_bad_test   = (~good_mask).astype(int)
baseline_bad = y_bad_test.mean()
print(f"Baseline bad-pose rate: {baseline_bad:.3f}  ({baseline_bad*100:.1f}%)")

best_thresholds = {}  # {k: {feat: best_thr}}

for k in K_VALUES:
    acts      = features_test[k]
    top_feats = top_spearman[k]
    best_thresholds[k] = {}

    fig, axes = plt.subplots(2, 5, figsize=(22, 8))
    fig.suptitle(f"k={k} — top 5 bad-pose features (Spearman ρ with RMSD), 4Å threshold",
                 fontsize=12)

    for col, feat in enumerate(top_feats):
        a      = acts[:, feat]
        active = a > 0
        a_act  = a[active]
        y_act  = y_bad_test[active]

        corr_r = all_feature_dfs[k].loc[all_feature_dfs[k]["feature"]==feat, "corr_rmsd"].values[0]

        print(f"\nk={k}, Feature {feat} (ρ={corr_r:.3f}): "
              f"{active.sum():,} active ({active.mean()*100:.1f}%)")
        print(f"  Among active: {y_act.mean()*100:.1f}% bad  "
              f"(baseline: {baseline_bad*100:.1f}%)")
        print(f"  Range [{a_act.min():.2f}, {a_act.max():.2f}]  mean={a_act.mean():.2f}")

        # Row 0: log-density distributions (active samples only)
        ax = axes[0, col]
        bins = np.linspace(0, a_act.max(), 60)
        ax.hist(a_act[y_act==0], bins=bins, density=True, alpha=0.6,
                color="mediumseagreen", label=f"good ({(y_act==0).sum():,})")
        ax.hist(a_act[y_act==1], bins=bins, density=True, alpha=0.6,
                color="salmon",        label=f"bad ({(y_act==1).sum():,})")
        ax.set_yscale("log")
        ax.set_xlabel("Activation value")
        ax.set_ylabel("Log density")
        ax.set_title(f"Feature {feat}  (ρ={corr_r:.3f})\n(active samples only)")
        ax.legend(fontsize=8)

        # Row 1: threshold sweep
        thresholds = np.linspace(a_act.min(), np.percentile(a_act, 99), 200)
        precisions, recalls, f1s = [], [], []
        for thr in thresholds:
            flagged = a >= thr
            if flagged.sum() == 0: continue
            prec = y_bad_test[flagged].mean()
            rec  = y_bad_test[flagged].sum() / y_bad_test.sum()
            f1   = 2*prec*rec / (prec+rec+1e-9)
            precisions.append(prec); recalls.append(rec); f1s.append(f1)

        best_idx = int(np.argmax(f1s))
        best_thr = thresholds[best_idx]
        best_thresholds[k][feat] = best_thr

        ax2 = axes[1, col]
        t = thresholds[:len(f1s)]
        ax2.plot(t, precisions, color="steelblue",  lw=2, label="precision")
        ax2.plot(t, recalls,    color="darkorange", lw=2, label="recall")
        ax2.plot(t, f1s,        color="purple",     lw=2, label="F1")
        ax2.axhline(baseline_bad, color="grey", linestyle="--", lw=1,
                    label=f"baseline {baseline_bad:.2f}")
        ax2.axvline(best_thr, color="red", linestyle=":", lw=1.5,
                    label=f"best-F1 thr={best_thr:.2f}")
        ax2.set_xlabel("Activation threshold")
        ax2.set_ylabel("Score")
        ax2.set_title(f"Feature {feat} — threshold sweep")
        ax2.legend(fontsize=7)
        ax2.set_ylim(0, 1.05)

        best_flagged = (a >= best_thr)
        print(f"  Best-F1 thr={best_thr:.3f}  "
              f"prec={precisions[best_idx]:.3f}  "
              f"rec={recalls[best_idx]:.3f}  "
              f"F1={f1s[best_idx]:.3f}  "
              f"flagged={best_flagged.sum():,}")

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"threshold_sweep_k{k}.png"),
                dpi=150, bbox_inches="tight")
    plt.show()


Top 5 by Spearman: {3: [107, 70, 80, 88, 92], 8: [53, 103, 26, 105, 46], 15: [73, 38, 67, 2, 85]}
Baseline bad-pose rate: 0.634  (63.4%)

k=3, Feature 107 (ρ=0.360): 6,151 active (9.0%)
  Among active: 96.0% bad  (baseline: 63.4%)
  Range [0.46, 9.32]  mean=4.20
  Best-F1 thr=0.456  prec=0.960  rec=0.136  F1=0.238  flagged=6,151

k=3, Feature 70 (ρ=0.212): 1,606 active (2.3%)
  Among active: 100.0% bad  (baseline: 63.4%)
  Range [1.24, 8.13]  mean=6.05
  Best-F1 thr=1.241  prec=1.000  rec=0.037  F1=0.071  flagged=1,606

k=3, Feature 80 (ρ=0.199): 13,505 active (19.7%)
  Among active: 70.4% bad  (baseline: 63.4%)
  Range [0.60, 10.43]  mean=3.70


/opt/schrodinger/suites2025-3/internal/lib/python3.11/site-packages/numpy/lib/histograms.py:883: RuntimeWarning: invalid value encountered in divide
  return n/db/n.sum(), bin_edges


  Best-F1 thr=0.601  prec=0.704  rec=0.219  F1=0.334  flagged=13,505

k=3, Feature 88 (ρ=0.163): 5,010 active (7.3%)
  Among active: 72.8% bad  (baseline: 63.4%)
  Range [1.06, 6.40]  mean=3.70
  Best-F1 thr=1.081  prec=0.728  rec=0.084  F1=0.150  flagged=5,008

k=3, Feature 92 (ρ=0.154): 4,312 active (6.3%)
  Among active: 78.1% bad  (baseline: 63.4%)
  Range [1.14, 10.50]  mean=4.56
  Best-F1 thr=1.308  prec=0.782  rec=0.077  F1=0.141  flagged=4,309

k=8, Feature 53 (ρ=0.416): 7,976 active (11.6%)
  Among active: 94.1% bad  (baseline: 63.4%)
  Range [0.87, 5.80]  mean=2.91
  Best-F1 thr=0.896  prec=0.941  rec=0.173  F1=0.292  flagged=7,973

k=8, Feature 103 (ρ=0.307): 15,319 active (22.3%)
  Among active: 80.5% bad  (baseline: 63.4%)
  Range [0.47, 7.75]  mean=3.49
  Best-F1 thr=0.468  prec=0.805  rec=0.283  F1=0.419  flagged=15,319

k=8, Feature 26 (ρ=0.265): 2,914 active (4.2%)
  Among active: 98.7% bad  (baseline: 63.4%)
  Range [0.76, 5.02]  mean=2.57
  Best-F1 thr=0.757  prec=0.

In [29]:
from itertools import combinations

ablation_results = {}

for k in K_VALUES:
    acts      = features_test[k]
    top_feats = top_spearman[k]
    flags     = {f: (acts[:, f] >= best_thresholds[k][f]).astype(int) for f in top_feats}

    records = []
    for r in range(1, len(top_feats)+1):
        for combo in combinations(top_feats, r):
            combined = np.zeros(len(y_bad_test), dtype=int)
            for feat in combo: combined |= flags[feat]
            flagged = combined.astype(bool)
            if flagged.sum() == 0: continue
            prec      = y_bad_test[flagged].mean()
            rec       = y_bad_test[flagged].sum() / y_bad_test.sum()
            f1        = 2*prec*rec / (prec+rec+1e-9)
            good_kept = (~flagged)[y_bad_test==0].mean()
            records.append({
                "subset":      "+".join(str(f) for f in combo),
                "n_flagged":   int(flagged.sum()),
                "pct_flagged": flagged.mean()*100,
                "precision":   prec,
                "recall":      rec,
                "f1":          f1,
                "good_kept":   good_kept,
            })

    abl_df = pd.DataFrame(records)
    ablation_results[k] = abl_df
    abl_df.to_csv(os.path.join(OUTPUT_DIR, f"ablation_k{k}.csv"), index=False)

    print(f"\n=== k={k} ablation (4Å threshold) ===")
    print(f"Baseline bad-pose rate: {baseline_bad:.3f}")
    print(f"{'subset':<25} {'n_flag':>7} {'%flag':>6} "
          f"{'prec':>7} {'recall':>7} {'F1':>7} {'good_kept':>10}")
    print("-" * 73)
    for _, row in abl_df.iterrows():
        print(f"{row['subset']:<25} {row['n_flagged']:>7,} "
              f"{row['pct_flagged']:>6.1f}% "
              f"{row['precision']:>7.3f} {row['recall']:>7.3f} "
              f"{row['f1']:>7.3f} {row['good_kept']:>10.3f}")
    print(f"  Saved: ablation_k{k}.csv")



=== k=3 ablation (4Å threshold) ===
Baseline bad-pose rate: 0.634
subset                     n_flag  %flag    prec  recall      F1  good_kept
-------------------------------------------------------------------------
107                         6,151    9.0%   0.960   0.136   0.238      0.990
70                          1,606    2.3%   1.000   0.037   0.071      1.000
80                         13,505   19.7%   0.704   0.219   0.334      0.841
88                          5,008    7.3%   0.728   0.084   0.150      0.946
92                          4,309    6.3%   0.782   0.077   0.141      0.963
107+70                      7,741   11.3%   0.968   0.172   0.293      0.990
107+80                     16,934   24.7%   0.751   0.292   0.421      0.832
107+88                      8,736   12.7%   0.823   0.165   0.275      0.939
107+92                     10,457   15.2%   0.887   0.213   0.344      0.953
70+80                      13,803   20.1%   0.711   0.226   0.343      0.841
70+88        

In [30]:
# Individual feature filter plots — one figure per k
# For each feature: % bad poses flagged (recall) and % good poses flagged (false alarm)
# plus precision shown as a marker

for k in K_VALUES:
    acts      = features_test[k]
    top_feats = top_spearman[k]
    n         = len(top_feats)

    # Compute per-feature stats at best-F1 threshold
    rows = []
    for feat in top_feats:
        thr     = best_thresholds[k][feat]
        flagged = acts[:, feat] >= thr
        n_bad   = y_bad_test.sum()
        n_good  = (~good_mask).size - n_bad
        bad_flagged  = (flagged &  y_bad_test.astype(bool)).sum()
        good_flagged = (flagged & ~y_bad_test.astype(bool)).sum()
        bad_kept     = (~flagged &  y_bad_test.astype(bool)).sum()
        good_kept_n  = (~flagged & ~y_bad_test.astype(bool)).sum()
        corr_r = all_feature_dfs[k].loc[
            all_feature_dfs[k]["feature"]==feat, "corr_rmsd"].values[0]
        rows.append({
            "feat":          feat,
            "corr_r":        corr_r,
            "pct_bad_flagged":  bad_flagged  / n_bad  * 100,
            "pct_bad_kept":     bad_kept     / n_bad  * 100,
            "pct_good_flagged": good_flagged / (len(y_bad_test) - n_bad) * 100,
            "pct_good_kept":    good_kept_n  / (len(y_bad_test) - n_bad) * 100,
            "precision": y_bad_test[flagged].mean() if flagged.sum() > 0 else 0,
            "f1": ablation_results[k].loc[
                ablation_results[k]["subset"]==str(feat), "f1"].values[0]
                  if str(feat) in ablation_results[k]["subset"].values else 0,
        })

    fig, axes = plt.subplots(1, n, figsize=(4*n, 5), sharey=False)
    fig.suptitle(f"k={k} — individual feature filter breakdown (4Å threshold)",
                 fontsize=12)

    for ax, row in zip(axes, rows):
        feat = row["feat"]
        categories = ["Bad poses", "Good poses"]
        flagged_pct = [row["pct_bad_flagged"], row["pct_good_flagged"]]
        kept_pct    = [row["pct_bad_kept"],    row["pct_good_kept"]]

        x = np.arange(2)
        w = 0.35
        bars_flag = ax.bar(x - w/2, flagged_pct, w,
                           color=["#e05c5c", "#f5a623"], label="Flagged (removed)")
        bars_kept = ax.bar(x + w/2, kept_pct, w,
                           color=["#a8d8a8", "#a8c8f0"], label="Kept")

        # Annotate bars
        for bar in list(bars_flag) + list(bars_kept):
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.8,
                    f"{h:.1f}%", ha="center", va="bottom", fontsize=8)

        ax.set_xticks(x)
        ax.set_xticklabels(categories, fontsize=9)
        ax.set_ylim(0, 115)
        ax.set_ylabel("% of poses")
        ax.set_title(
            f"Feature {feat}\n"
            f"ρ={row['corr_r']:.3f}  prec={row['precision']:.3f}  F1={row['f1']:.3f}",
            fontsize=9
        )
        ax.legend(fontsize=7, loc="upper right")
        ax.axhline(100, color="grey", linestyle=":", lw=0.8, alpha=0.5)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"feature_filter_breakdown_k{k}.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: feature_filter_breakdown_k{k}.png")


Saved: feature_filter_breakdown_k3.png
Saved: feature_filter_breakdown_k8.png
Saved: feature_filter_breakdown_k15.png


### 8. Summary

In [31]:
output_files = [
    "metadata_rmsd4.csv",
    "features_for_paper.npz",
    "top_features_k3.csv",
    "top_features_k8.csv",
    "top_features_k15.csv",
    "roc_auc_comparison.png",
    "threshold_sweep_k3.png",
    "threshold_sweep_k8.png",
    "threshold_sweep_k15.png",
    "ablation_k3.csv",
    "ablation_k8.csv",
    "ablation_k15.csv",
    "ablation_plot_k3.png",
    "ablation_plot_k8.png",
    "ablation_plot_k15.png",
]
print("Output files:")
for fname in output_files:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1024
        print(f"  {fname:<38s}  {size:8.1f} KB")
    else:
        print(f"  {fname:<38s}  (not yet generated)")

print(f"\nThreshold: {RMSD_THRESHOLD}Å  |  Test set: {len(test_idx):,} samples")
print("\nROC-AUC summary:")
for name, auc in roc_results.items():
    print(f"  {name:<30}  auPR={aupr_results[name]:.4f}  AUC={auc:.4f}")


Output files:
  metadata_rmsd4.csv                       31404.1 KB
  features_for_paper.npz                    9334.7 KB
  top_features_k3.csv                          2.4 KB
  top_features_k8.csv                          2.4 KB
  top_features_k15.csv                         2.4 KB
  roc_auc_comparison.png                     141.1 KB
  threshold_sweep_k3.png                     209.0 KB
  threshold_sweep_k8.png                     218.1 KB
  threshold_sweep_k15.png                    251.3 KB
  ablation_k3.csv                              3.3 KB
  ablation_k8.csv                              3.4 KB
  ablation_k15.csv                             3.3 KB
  ablation_plot_k3.png                       152.5 KB
  ablation_plot_k8.png                       161.6 KB
  ablation_plot_k15.png                      154.5 KB

Threshold: 4.0Å  |  Test set: 68,614 samples

ROC-AUC summary:
  Raw latents (30-dim)            auPR=0.6125  AUC=0.7146
  SAE k=3  (120-dim)              auPR=0.3825  AUC=0.5